# ReasonGuard v0.2 — Multi-Bench Combined Training

**Notebook 34 · OpenInterp · 2026-04-29**

Trains the production ReasonGuard probe using **FabricationGuard methodology**: combined training set across all 3 reasoning benches (GSM8K + StrategyQA + MATH), single probe at L55/mid_think.

## Why this exists

ReasonGuard v0.1 (notebook 32) trained on GSM8K alone and **failed cross-bench** (AUROC 0.605 on StrategyQA vs 0.888 within). Hypothesis: domain-bound signal.

FabricationGuard solved the same problem by training multi-bench from the start (TruthfulQA + HaluEval + MMLU + SimpleQA), achieving AUROC 0.882 cross-bench. v0.2 applies the same recipe to reasoning-faithfulness.

## Inputs

`rollouts.npz` from notebook 32 v2, persisted at:
```
/content/drive/MyDrive/openinterp_runs/32_reasoningguard_v2/rollouts.npz
```
Contains 650 rollouts × 12 (layer, position) combos of residuals + labels.

## Output

- Probe v0.2 (`probe.joblib` with combined-train weights)
- Cross-bench AUROC table (per-bench held-out)
- Comparison with v0.1 numbers
- HF push to `caiovicentino1/ReasoningGuard-linearprobe-qwen36-27b` v0.2 branch

## Compute

CPU-only (sklearn). ~5 min on Colab free tier. No GPU needed.


## 0. Drive mount + checkpoint dir (non-negotiable)


In [ ]:
# === DRIVE MOUNT — non-negotiable for any run >30min ===
from pathlib import Path
import os, sys

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print(f"Drive mount FAILED: {e}"); raise

DRIVE_ROOT = Path("/content/drive/MyDrive")
assert DRIVE_ROOT.exists(), "Drive mount silently failed"
NB_NAME = "34_reasonguard_v0_2_multibench"
OUT = DRIVE_ROOT / "openinterp_runs" / NB_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "_dry_run.txt").write_text("drive mount OK")
print(f"✓ Drive checkpoint dir: {OUT}")
print(f"  Contents: {sorted(p.name for p in OUT.iterdir())}")


## 1. Setup


In [ ]:
%pip install -q -U scikit-learn numpy pandas matplotlib joblib huggingface_hub
import numpy as np, json, joblib, time
from pathlib import Path
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

CFG = {
    "rollouts_path": str(DRIVE_ROOT / "openinterp_runs" / "32_reasoningguard_v2" / "rollouts.npz"),
    "probe_layer":   55,
    "probe_position":"mid_think",
    "lr_C_sweep":    [0.001, 0.01, 0.1, 1.0, 10.0],
    "random_seed":   42,
    "train_size":    0.7,
    "hf_results_repo": "caiovicentino1/ReasoningGuard-linearprobe-qwen36-27b",
}
print(json.dumps(CFG, indent=2, default=str))


## 2. Load rollouts from notebook 32


In [ ]:
rollouts = np.load(CFG["rollouts_path"], allow_pickle=True)
print("Available arrays:", sorted(rollouts.files)[:20], "...")

# Reconstruct data dict — schema: y_<bench>, res_<bench>_<pos>_L<layer>
benches = sorted({k.split("_",1)[1] for k in rollouts.files if k.startswith("y_")})
print(f"Benches found: {benches}")
for b in benches:
    y = rollouts[f"y_{b}"]
    print(f"  {b}: n={len(y)}, halu_rate={100*y.mean():.1f}%")

L, POS = CFG["probe_layer"], CFG["probe_position"]
data = {}
for b in benches:
    key = f"res_{b}_{POS}_L{L}"
    if key not in rollouts.files:
        print(f"  ⚠️  {key} missing — bench {b} skipped")
        continue
    data[b] = {"X": rollouts[key], "y": rollouts[f"y_{b}"]}


## 3. Combined-train probe (FabricationGuard methodology)


In [ ]:
# 70/30 split per bench, stratified by label
splits = {}
for b, d in data.items():
    if len(np.unique(d["y"])) < 2:
        print(f"  {b}: only one class — skipping"); continue
    idx_tr, idx_te = train_test_split(
        np.arange(len(d["y"])), test_size=1-CFG["train_size"],
        stratify=d["y"], random_state=CFG["random_seed"])
    splits[b] = {"tr": idx_tr, "te": idx_te}
    print(f"  {b}: {len(idx_tr)} train, {len(idx_te)} test")

X_train_combined = np.concatenate([data[b]["X"][splits[b]["tr"]] for b in splits])
y_train_combined = np.concatenate([data[b]["y"][splits[b]["tr"]] for b in splits])
print(f"\nCombined train: n={len(y_train_combined)}, halu_rate={100*y_train_combined.mean():.1f}%")

scaler = StandardScaler().fit(X_train_combined)
clf = LogisticRegressionCV(
    Cs=CFG["lr_C_sweep"], cv=5, penalty="l2", solver="lbfgs",
    max_iter=2000, scoring="roc_auc", n_jobs=-1, refit=True,
).fit(scaler.transform(X_train_combined), y_train_combined)
print(f"Best C: {float(clf.C_[0])}")


## 4. Per-bench held-out evaluation


In [ ]:
results = {}
for b, sp in splits.items():
    X_te, y_te = data[b]["X"][sp["te"]], data[b]["y"][sp["te"]]
    scores = clf.predict_proba(scaler.transform(X_te))[:, 1]
    auc = roc_auc_score(y_te, scores)
    results[b] = {"auroc": float(auc), "n_test": len(y_te)}
    badge = "✅" if auc >= 0.70 else "🟡" if auc >= 0.60 else "❌"
    print(f"  {b:12s}: AUROC = {auc:.3f} (n={len(y_te)})  {badge}")

# Comparison with v0.1
v01 = {"gsm8k_within": 0.888, "strategyqa_cross": 0.605}
print("\n=== v0.1 vs v0.2 ===")
print(f"  v0.1 GSM8K within (single-bench train):     {v01['gsm8k_within']:.3f}")
print(f"  v0.1 StrategyQA cross (single-bench train): {v01['strategyqa_cross']:.3f}")
print(f"  v0.2 GSM8K held-out (combined-train):       {results.get('gsm8k', {}).get('auroc', float('nan')):.3f}")
print(f"  v0.2 StrategyQA held-out (combined-train):  {results.get('strategyqa', {}).get('auroc', float('nan')):.3f}")
if "math" in results:
    print(f"  v0.2 MATH held-out (combined-train):        {results['math']['auroc']:.3f}")

# Verdict
all_aurocs = [r["auroc"] for r in results.values()]
mean_auc = np.mean(all_aurocs)
all_passed = all(a >= 0.70 for a in all_aurocs)
print(f"\nMean AUROC: {mean_auc:.3f}")
print(f"All benches ≥ 0.70: {'✅' if all_passed else '❌'}")


## 5. Save probe v0.2 + push to HF


In [ ]:
joblib.dump({
    "probe": clf, "scaler": scaler,
    "layer": CFG["probe_layer"], "position": CFG["probe_position"],
    "C": float(clf.C_[0]),
    "training": "combined-bench (gsm8k + strategyqa + math)",
    "version": "v0.2",
}, OUT / "probe_v02.joblib")

verdict = {
    "version": "v0.2",
    "date": time.strftime("%Y-%m-%d"),
    "methodology": "FabricationGuard-style combined-bench training",
    "training_benches": list(splits.keys()),
    "per_bench_auroc": results,
    "mean_auroc": float(np.mean([r["auroc"] for r in results.values()])),
    "all_passed_70": all(r["auroc"] >= 0.70 for r in results.values()),
    "comparison_v01": v01,
    "config": {k: v for k, v in CFG.items() if k != "rollouts_path"},
}
(OUT / "verdict_v02.json").write_text(json.dumps(verdict, indent=2, default=str))
print(json.dumps(verdict, indent=2))

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_folder(
        folder_path=str(OUT), repo_id=CFG["hf_results_repo"],
        repo_type="dataset", token=HF_TOKEN,
        commit_message=f"ReasonGuard v0.2 — combined-train probe ({time.strftime('%Y-%m-%d')})",
        path_in_repo="v0.2/",
    )
    print(f"\n✅ Pushed to https://huggingface.co/datasets/{CFG['hf_results_repo']}/tree/main/v0.2")


## 6. Honest interpretation

**If all benches ≥ 0.70**: ReasonGuard v0.2 generalizes across reasoning domains. Ship as `live` on ProbeBench, replace v0.1 entry. Methodology validated: multi-bench training is necessary for cross-domain transfer in reasoning probes.

**If some benches < 0.70**: partial generalization. Ship v0.2 with narrow-scope tagline like FabricationGuard's MMLU caveat (out-of-scope rather than failed). Honest registration of partial signal.

**If most benches < 0.70**: methodology insufficient — single-layer + single-position is too restrictive. Pivot to multi-layer probe ensemble (v0.3 candidate).

Either way, both v0.1 and v0.2 numbers stay published on ProbeBench. The framework honors honest negative results.
